In [ ]:
import openmeteo_requests
import requests
import requests_cache
import os
import json
from retry_requests import retry
from datetime import datetime

In [2]:
OUTPUT_DIR = "../../data/raw/"
CURRENT_YEAR = datetime.now().year
YEARS_TO_TRY = [CURRENT_YEAR, CURRENT_YEAR - 1, CURRENT_YEAR - 2]

In [3]:
def accidents_for_year(year):

    url = "https://opendata-ajuntament.barcelona.cat/data/api/action/datastore_search"

    params = {
        "resource_id": "8cfddcbe-3403-4a6c-8897-c13238da900e",
        "filters": json.dumps({"Nk_Any": str(year)})
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()

    records = data["result"]["records"]

    return records

In [4]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

for year in YEARS_TO_TRY:
    print(f"Probando año {year}...")
    records = accidents_for_year(year)

    if records:

        print(f"Datos encontrados para {year}: {len(records)} registros")

        output_path_acc = os.path.join(
            OUTPUT_DIR, f"accidents_bcn_{year}.json"
        )

        with open(output_path_acc, "w", encoding="utf-8") as f:
            json.dump(records, f, ensure_ascii=False, indent=2)

        print(f"Guardado en {output_path_acc}")
        break
else:
    raise RuntimeError("No se encontraron datos en ningún año probado")

Probando año 2025...
Probando año 2024...
Datos encontrados para 2024: 100 registros
Guardado en ../../data/raw/accidents_bcn_2024.json


In [5]:
start_date = datetime(year, 1, 1).strftime("%Y-%m-%d")
end_date = datetime(year, 12, 31).strftime("%Y-%m-%d")

In [6]:
# setup the open-meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries= 5, backoff_factor= 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [7]:
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 41.38,
	"longitude": 2.15,
    "start_date": start_date,
    "end_date": end_date,
	"hourly": "temperature_2m,precipitation,weathercode,windspeed_10m",
    "timezone": "Europe/Madrid"
}

response = openmeteo.weather_api(url, params=params)[0]

hourly = response.Hourly()

In [ ]:
# Process hourly data.

raw_weather = {
    "start_time": hourly.Time(),
    "end_time": hourly.TimeEnd(),
    "interval_seconds": hourly.Interval(),
    "hourly": {
        "temperature_2m": hourly.Variables(0).ValuesAsNumpy().tolist(),
        "precipitation": hourly.Variables(1).ValuesAsNumpy().tolist(),
        "weathercode": hourly.Variables(2).ValuesAsNumpy().tolist(),
        "windspeed_10m": hourly.Variables(3).ValuesAsNumpy().tolist()
    }
}

output_path_hd = os.path.join(
            OUTPUT_DIR, f"weather_bcn_{year}.json"
        )

with open(output_path_acc, "w", encoding="utf-8") as f:
            json.dump(raw_weather, f, ensure_ascii=False, indent=2)

print(f"Guardado en {output_path_hd}")

Guardado en ../../data/raw/weather_bcn_2024.json
